In [ ]:
# Extracting embedding in order to do UMAP on them

# If running in a clean environment, uncomment the next lines to install packages.
# You can re-run this cell safely; it will skip installs if already present.

# !pip install --quiet torch
# Try the officialSM library
# !pip install --quiet fair-esm
# If the above doesn't work in your environment, try:
#!pip install --quiet esm
# Progress bar
#!pip install --quiet tqdm pandas pyarrow

import os
import re
import gc
import json
import numpy as np
import pandas as pd
import torch

# ESM import (works with 'fair-esm' or 'esm' package)
from esm import pretrained
from tqdm.auto import tqdm

In [ ]:
# Path to your CSV (relative to the notebook location)
CSV_PATH = "1_Data/preprocessed_data.csv"

df = pd.read_csv(CSV_PATH)

# Basic sanity check
expected_cols = {"Clone name", "VH Protein", "VL Protein"}
missing = expected_cols - set(df.columns)
assert not missing, f"Missing column(s) in CSV: {missing}"

print(f"Loaded {len(df)} rows from {CSV_PATH}")
df.head(3)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load the pretrained ESM-2 150M parameter model and its alphabet
model, alphabet = pretrained.esm2_t30_150M_UR50D()
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

EMBED_DIM = model.embed_dim  # should be 640 for t30_150M
N_LAYERS = model.num_layers  # we will use the final layer representations
print(f"Model embed_dim: {EMBED_DIM}, num_layers: {N_LAYERS}")

In [ ]:
import torch
from esm import pretrained

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Works across esm/fair-esm versions:
model, alphabet = pretrained.load_model_and_alphabet_hub("esm2_t30_150M_UR50D")

model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

EMBED_DIM = model.embed_dim
N_LAYERS = model.num_layers
print(f"Model embed_dim: {EMBED_DIM}, num_layers: {N_LAYERS}")

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Make sure your torch hub cache is writable (optional)
# torch.hub.set_dir(r"C:\MasterThesisTechnicalPart\.torch_cache")  # e.g., on Windows

model, alphabet = torch.hub.load(
    repo_or_dir="facebookresearch/esm:main",
    model="esm2_t30_150M_UR50D",
)
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

EMBED_DIM = model.embed_dim
N_LAYERS = model.num_layers
print(f"Model embed_dim: {EMBED_DIM}, num_layers: {N_LAYERS}")

In [ ]:
@torch.no_grad()
def embed_sequences_mean(seqs, names=None, batch_size=32):
    """
    Args:
        seqs: list[str] amino-acid sequences
        names: optional list[str] sequence identifiers
        batch_size: number of sequences per forward pass (adjust to memory)

    Returns:
        np.ndarray of shape (N, EMBED_DIM) with per-sequence embeddings
    """
    if names is None:
        names = [f"seq_{i}" for i in range(len(seqs))]
        
    # Prepare data tuples for ESM batch converter
    all_data = list(zip(names, seqs))
    outputs = np.zeros((len(seqs), EMBED_DIM), dtype=np.float32)

    start = 0
    for i in tqdm(range(0, len(all_data), batch_size), desc="Embedding"):
        batch = all_data[i:i+batch_size]
        batch_labels, batch_strs, batch_tokens = batch_converter(batch)
        batch_tokens = batch_tokens.to(device)

        # Forward pass: request only the final layer
        results = model(batch_tokens, repr_layers=[N_LAYERS], return_contacts=False)
        token_reps = results["representations"][N_LAYERS]  # (B, T, D)

        # For each sequence, average representations over actual residues
        # Positions: [0]=BOS, [1:seq_len+1]=residues, [seq_len+1]=EOS
        # So we select [1:tokens_len-1]
        for b_idx, toks in enumerate(batch_tokens):
            tokens_len = (toks != alphabet.padding_idx).sum().item()
            rep = token_reps[b_idx, 1:tokens_len-1].mean(0)  # (D,)
            outputs[start + b_idx] = rep.cpu().float().numpy()

        start += len(batch)

        # Housekeeping
        del results, token_reps, batch_tokens
        torch.cuda.empty_cache() if device == "cuda" else None

    return outputs

In [ ]:
# You can tweak batch_size based on GPU/CPU memory. CPU works fine with 8–16.
BATCH_SIZE = 32 if device == "cuda" else 8

vh_embeddings = embed_sequences_mean(vh_sequences, names=[f"VH:{n}" for n in clone_names],
                                     batch_size=BATCH_SIZE)
vl_embeddings = embed_sequences_mean(vl_sequences, names=[f"VL:{n}" for n in clone_names],
                                     batch_size=BATCH_SIZE)

print("VH embeddings shape:", vh_embeddings.shape)
print("VL embeddings shape:", vl_embeddings.shape)
assert vh_embeddings.shape == vl_embeddings.shape == (len(df), EMBED_DIM)

In [ ]:
OUT_NPZ = "../1_Data/esm2_t30_150M_embeddings_vh_vl.npz"
np.savez_compressed(
    OUT_NPZ,
    model="esm2_t30_150M_UR50D",
    embed_dim=EMBED_DIM,
    clone_names=np.array(clone_names, dtype=object),
    vh_embeddings=vh_embeddings,
    vl_embeddings=vl_embeddings,
)
print(f"Saved NPZ: {OUT_NPZ}")

# Optional: tidy tabular formats (one row per clone)
vh_df = pd.DataFrame(vh_embeddings, index=clone_names,
                     columns=[f"vh_{i}" for i in range(EMBED_DIM)]).reset_index().rename(columns={"index": "Clone name"})
vl_df = pd.DataFrame(vl_embeddings, index=clone_names,
                     columns=[f"vl_{i}" for i in range(EMBED_DIM)]).reset_index().rename(columns={"index": "Clone name"})

# Parquet (fast, compact)
VH_PARQUET = "../1_Data/esm2_t30_150M_vh_embeddings.parquet"
VL_PARQUET = "../1_Data/esm2_t30_150M_vl_embeddings.parquet"
vh_df.to_parquet(VH_PARQUET, index=False)
vl_df.to_parquet(VL_PARQUET, index=False)
print(f"Saved Parquet:\n  {VH_PARQUET}\n  {VL_PARQUET}")

# CSV (bigger, but universally readable)
VH_CSV = "../1_Data/esm2_t30_150M_vh_embeddings.csv"
VL_CSV = "../1_Data/esm2_t30_150M_vl_embeddings.csv"
vh_df.to_csv(VH_CSV, index=False)
vl_df.to_csv(VL_CSV, index=False)
print(f"Saved CSV:\n  {VH_CSV}\n  {VL_CSV}")